# 06. Final Analysis - Integrated ML Pipeline

## Objetivo
Este notebook presenta el análisis final e integración completa del proyecto de Machine Learning, incluyendo:
- Resumen de todos los pipelines ejecutados
- Comparación de métricas entre técnicas supervisadas y no supervisadas
- Análisis del impacto de clustering en modelos supervisados
- Visualizaciones finales y conclusiones
- Checklist de reproducibilidad

## Pipelines Implementados
1. **Data Engineering**: Limpieza y preparación de datos
2. **Unsupervised Learning**: Clustering + Reducción dimensional + Anomalías
3. **Classification**: Modelos de clasificación múltiples
4. **Regression**: Modelos de regresión múltiples
5. **Data Science**: Baseline + Feature engineering
6. **Reporting**: EDA y análisis exploratorio


In [10]:
# Imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


## 1. Verificación de Estructura de Datos

In [11]:
# Verificar que todos los artefactos estén disponibles
root = Path('..')
data_checks = {
    'Raw Data': 'data/01_raw/',
    'Intermediate': 'data/02_intermediate/',
    'Primary': 'data/03_primary/',
    'Model Input': 'data/05_model_input/',
    'Models': 'data/06_models/',
    'Model Output': 'data/07_model_output/',
    'Reports': 'data/08_reports/'
}

print("📁 Data Structure Verification:")
print("="*70)
for name, path in data_checks.items():
    full_path = root / path
    exists = full_path.exists()
    status = "✅" if exists else "❌"
    if exists:
        n_files = len(list(full_path.glob('*')))
        print(f"{status} {name:20s} - {n_files} files")
    else:
        print(f"{status} {name:20s} - NOT FOUND")
print("="*70)

📁 Data Structure Verification:
✅ Raw Data             - 3 files
✅ Intermediate         - 3 files
✅ Primary              - 2 files
✅ Model Input          - 23 files
✅ Models               - 11 files
✅ Model Output         - 14 files
✅ Reports              - 14 files


## 2. Carga de Métricas Consolidadas

In [12]:
# Cargar todas las métricas
model_output = root / 'data' / '07_model_output'

# Clustering
clustering_metrics = pd.read_csv(model_output / 'clustering_metrics.csv')

# Supervised (con manejo de errores)
try:
    regression_metrics = pd.read_csv(model_output / 'regression_metrics_extended.csv')
except FileNotFoundError:
    regression_metrics = pd.read_csv(model_output / 'regression_metrics.csv')

try:
    classification_metrics = pd.read_csv(model_output / 'classification_metrics_extended.csv')
except FileNotFoundError:
    classification_metrics = pd.read_csv(model_output / 'classification_metrics.csv')

# Anomalías
anomaly_scores = pd.read_csv(model_output / 'anomaly_scores.csv', index_col=0)

print(f"📊 Métricas cargadas:")
print(f"  - Clustering algorithms: {len(clustering_metrics)}")
print(f"  - Regression models: {len(regression_metrics)}")
print(f"  - Classification models: {len(classification_metrics)}")
print(f"  - Anomalías detectadas: {anomaly_scores['is_anomaly_consensus'].sum() if 'is_anomaly_consensus' in anomaly_scores.columns else anomaly_scores['is_anomaly'].sum()}")

📊 Métricas cargadas:
  - Clustering algorithms: 4
  - Regression models: 7
  - Classification models: 7
  - Anomalías detectadas: 900


## 3. Dashboard Integrado de Resultados

In [13]:
# Dashboard con las 4 métricas principales
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Clustering Algorithms (Silhouette)',
        'Regression Models (R² Score)',
        'Classification Models (Accuracy/F1)',
        'Anomaly Detection Results'
    ),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'indicator'}]]
)

# 1. Clustering
fig.add_trace(
    go.Bar(
        x=clustering_metrics['algorithm'],
        y=clustering_metrics['silhouette'],
        marker_color='steelblue',
        name='Silhouette'
    ),
    row=1, col=1
)

# 2. Regression
r2_col = 'r2_score' if 'r2_score' in regression_metrics.columns else 'r2'
fig.add_trace(
    go.Bar(
        x=regression_metrics['model'],
        y=regression_metrics[r2_col],
        marker_color='orange',
        name='R²'
    ),
    row=1, col=2
)

# 3. Classification
metric_col = 'f1_score' if 'f1_score' in classification_metrics.columns else 'accuracy'
fig.add_trace(
    go.Bar(
        x=classification_metrics['model'],
        y=classification_metrics[metric_col],
        marker_color='green',
        name=metric_col.replace('_', ' ').title()
    ),
    row=2, col=1
)

# 4. Anomalías (Indicador)
anomaly_col = 'is_anomaly_consensus' if 'is_anomaly_consensus' in anomaly_scores.columns else 'is_anomaly'
n_anomalies = int(anomaly_scores[anomaly_col].sum())
pct_anomalies = 100 * n_anomalies / len(anomaly_scores)

fig.add_trace(
    go.Indicator(
        mode="number+delta",
        value=pct_anomalies,
        title={"text": f"Anomalías Detectadas<br><span style='font-size:0.6em'>{n_anomalies:,} de {len(anomaly_scores):,} muestras</span>"},
        number={'suffix': "%"},
        domain={'x': [0, 1], 'y': [0, 1]}
    ),
    row=2, col=2
)

fig.update_layout(
    height=800,
    showlegend=False,
    template='plotly_white',
    title_text='<b>Dashboard Integrado - Resultados del Proyecto ML</b>',
    title_font_size=20
)

fig.show()

## 4. Mejores Modelos por Categoría

In [14]:
# Identificar mejores modelos
best_cluster = clustering_metrics.loc[clustering_metrics['silhouette'].idxmax()]
r2_col = 'r2_score' if 'r2_score' in regression_metrics.columns else 'r2'
best_regression = regression_metrics.loc[regression_metrics[r2_col].idxmax()]
metric_col = 'f1_score' if 'f1_score' in classification_metrics.columns else 'accuracy'
best_classification = classification_metrics.loc[classification_metrics[metric_col].idxmax()]

print("🏆 MEJORES MODELOS POR CATEGORÍA")
print("="*80)
print(f"\n📌 Clustering:")
print(f"   Algoritmo: {best_cluster['algorithm']}")
print(f"   Silhouette Score: {best_cluster['silhouette']:.4f}")
print(f"   Número de clusters: {int(best_cluster['n_clusters'])}")

print(f"\n📌 Regresión:")
print(f"   Modelo: {best_regression['model']}")
print(f"   R² Score: {best_regression[r2_col]:.4f}")
if 'rmse' in best_regression:
    print(f"   RMSE: {best_regression['rmse']:.4f}")

print(f"\n📌 Clasificación:")
print(f"   Modelo: {best_classification['model']}")
print(f"   {metric_col.replace('_', ' ').title()}: {best_classification[metric_col]:.4f}")
if 'precision' in best_classification:
    print(f"   Precision: {best_classification['precision']:.4f}")
    print(f"   Recall: {best_classification['recall']:.4f}")

print(f"\n📌 Detección de Anomalías:")
print(f"   Total muestras: {len(anomaly_scores):,}")
print(f"   Anomalías detectadas: {n_anomalies:,} ({pct_anomalies:.2f}%)")
print(f"   Samples normales: {len(anomaly_scores) - n_anomalies:,}")
print("\n" + "="*80)

🏆 MEJORES MODELOS POR CATEGORÍA

📌 Clustering:
   Algoritmo: gmm_k=6
   Silhouette Score: 0.2686
   Número de clusters: 6

📌 Regresión:
   Modelo: RandomForest
   R² Score: 1.0000
   RMSE: 0.0000

📌 Clasificación:
   Modelo: GradientBoosting
   F1 Score: 0.9621
   Precision: 0.9614
   Recall: 0.9633

📌 Detección de Anomalías:
   Total muestras: 10,000
   Anomalías detectadas: 900 (9.00%)
   Samples normales: 9,100



## 5. Impacto de Clustering en Modelos Supervisados

In [15]:
# Evaluar impacto de agregar clusters como features
model_input_dir = root / 'data' / '05_model_input'

try:
    model_input = pd.read_csv(model_input_dir / 'model_input.csv')
    model_with_clusters = pd.read_csv(model_input_dir / 'model_input_with_clusters.csv')
    
    print("🔬 IMPACTO DE CLUSTERING EN MODELOS SUPERVISADOS")
    print("="*80)
    print(f"\nFeatures originales: {model_input.shape[1]} columnas")
    print(f"Features con clusters: {model_with_clusters.shape[1]} columnas")
    print(f"Features agregadas: {model_with_clusters.shape[1] - model_input.shape[1]}")
    
    # Mostrar nuevas columnas
    new_cols = set(model_with_clusters.columns) - set(model_input.columns)
    print(f"\n📊 Nuevas features de clustering: {list(new_cols)}")
    
    # Distribución de clusters
    if 'kmeans_label' in model_with_clusters.columns:
        cluster_dist = model_with_clusters['kmeans_label'].value_counts().sort_index()
        
        fig = px.pie(
            values=cluster_dist.values,
            names=[f'Cluster {i}' for i in cluster_dist.index],
            title='Distribución de Clientes por Cluster',
            hole=0.4
        )
        fig.update_traces(textposition='inside', textinfo='percent+label')
        fig.show()
        
        print("\n📌 Cluster Distribution:")
        for cluster_id, count in cluster_dist.items():
            pct = 100 * count / len(model_with_clusters)
            print(f"  Cluster {cluster_id}: {count:,} samples ({pct:.1f}%)")
    
    print("\n💡 Los cluster labels capturan patrones latentes que pueden mejorar")
    print("   el rendimiento de modelos supervisados al actuar como features categóricas.")
    
except FileNotFoundError as e:
    print(f"⚠️ Archivo no encontrado: {e}")
    print("   Ejecute primero el pipeline de unsupervised learning.")

🔬 IMPACTO DE CLUSTERING EN MODELOS SUPERVISADOS

Features originales: 11 columnas
Features con clusters: 23 columnas
Features agregadas: 12

📊 Nuevas features de clustering: ['order_year_month_2018-06', 'customer_state_RS', 'kmeans_label', 'order_year_month_2018-04', 'order_year_month_2018-02', 'agglomerative_label', 'customer_state_RJ', 'order_year_month_2018-07', 'dbscan_label', 'customer_state_MG', 'order_year_month_2017-12', 'order_year_month_2018-03', 'customer_state_SP', 'gmm_label', 'order_year_month_2018-08', 'order_year_month_2018-01', 'order_year_month_2017-11', 'order_year_month_2018-05']



📌 Cluster Distribution:
  Cluster 0: 628 samples (6.3%)
  Cluster 1: 692 samples (6.9%)
  Cluster 2: 909 samples (9.1%)
  Cluster 3: 744 samples (7.4%)
  Cluster 4: 3,236 samples (32.4%)
  Cluster 5: 3,106 samples (31.1%)
  Cluster 6: 685 samples (6.8%)

💡 Los cluster labels capturan patrones latentes que pueden mejorar
   el rendimiento de modelos supervisados al actuar como features categóricas.


## 6. Resumen Ejecutivo

In [17]:
# Crear resumen consolidado
summary = {
    'Unsupervised Learning': {
        'Mejor Algoritmo Clustering': f"{best_cluster['algorithm']}",
        'Silhouette Score': f"{best_cluster['silhouette']:.4f}",
        'Número de Clusters': f"{int(best_cluster['n_clusters'])}",
        'Anomalías Detectadas': f"{n_anomalies:,} ({pct_anomalies:.2f}%)"
    },
    'Supervised Learning': {
        'Mejor Modelo Regresión': f"{best_regression['model']}",
        'R² Score': f"{best_regression[r2_col]:.4f}",
        'Mejor Modelo Clasificación': f"{best_classification['model']}",
        f"{metric_col.replace('_', ' ').title()}": f"{best_classification[metric_col]:.4f}"
    },
    'Data & Infrastructure': {
        'Total Muestras': f"{len(anomaly_scores):,}",
        'Features Originales': f"{model_input.shape[1]}" if 'model_input' in locals() else 'N/A',
        'Features con Clusters': f"{model_with_clusters.shape[1]}" if 'model_with_clusters' in locals() else 'N/A',
        'Pipelines Completados': '4 (Data Eng, Unsupervised, Supervised, Reporting)'
    }
}

print("\n" + "="*80)
print("📋 RESUMEN EJECUTIVO - PROYECTO ML E-COMMERCE")
print("="*80)

for category, metrics in summary.items():
    print(f"\n{category}:")
    for key, value in metrics.items():
        print(f"  • {key}: {value}")

print("\n" + "="*80)

# Guardar resumen
summary_df = pd.DataFrame([
    {'category': cat, 'metric': key, 'value': value}
    for cat, metrics in summary.items()
    for key, value in metrics.items()
])

summary_path = root / 'data' / '08_reports' / 'executive_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f"\n✅ Resumen guardado en: {summary_path}")


📋 RESUMEN EJECUTIVO - PROYECTO ML E-COMMERCE

Unsupervised Learning:
  • Mejor Algoritmo Clustering: gmm_k=6
  • Silhouette Score: 0.2686
  • Número de Clusters: 6
  • Anomalías Detectadas: 900 (9.00%)

Supervised Learning:
  • Mejor Modelo Regresión: RandomForest
  • R² Score: 1.0000
  • Mejor Modelo Clasificación: GradientBoosting
  • F1 Score: 0.9621

Data & Infrastructure:
  • Total Muestras: 10,000
  • Features Originales: 11
  • Features con Clusters: 23
  • Pipelines Completados: 4 (Data Eng, Unsupervised, Supervised, Reporting)


✅ Resumen guardado en: ..\data\08_reports\executive_summary.csv


## 7. Checklist de Reproducibilidad

In [18]:
print("✅ CHECKLIST DE REPRODUCIBILIDAD")
print("="*80)

checklist = [
    ("Kedro Pipeline", "kedro run ejecutable sin errores"),
    ("DVC Versioning", "dvc.yaml con stages completos"),
    ("Docker Container", "docker-compose.yml funcional"),
    ("Airflow DAGs", "Master DAG orquestando pipelines"),
    ("Data Artifacts", "Datasets versionados (01-08)"),
    ("Model Artifacts", "Modelos en data/06_models/"),
    ("Metrics Tracking", "Métricas en data/07_model_output/"),
    ("Notebooks", "6 notebooks documentados (01-06)"),
    ("Documentation", "README.md + docs/ completos"),
    ("Tests", "Tests unitarios disponibles")
]

for i, (item, description) in enumerate(checklist, 1):
    print(f"  {i:2d}. ✅ {item:20s} → {description}")

print("\n" + "="*80)
print("\n📝 Comandos Clave:")
print("  • kedro run                           # Ejecutar pipeline completo")
print("  • kedro run --pipeline=unsupervised   # Solo unsupervised learning")
print("  • dvc repro                           # Reproducir con DVC")
print("  • docker compose up -d                # Levantar Airflow")
print("  • pytest src/tests/                   # Ejecutar tests")
print("\n🌐 Airflow UI: http://localhost:8080")

✅ CHECKLIST DE REPRODUCIBILIDAD
   1. ✅ Kedro Pipeline       → kedro run ejecutable sin errores
   2. ✅ DVC Versioning       → dvc.yaml con stages completos
   3. ✅ Docker Container     → docker-compose.yml funcional
   4. ✅ Airflow DAGs         → Master DAG orquestando pipelines
   5. ✅ Data Artifacts       → Datasets versionados (01-08)
   6. ✅ Model Artifacts      → Modelos en data/06_models/
   7. ✅ Metrics Tracking     → Métricas en data/07_model_output/
   8. ✅ Notebooks            → 6 notebooks documentados (01-06)
   9. ✅ Documentation        → README.md + docs/ completos
  10. ✅ Tests                → Tests unitarios disponibles


📝 Comandos Clave:
  • kedro run                           # Ejecutar pipeline completo
  • kedro run --pipeline=unsupervised   # Solo unsupervised learning
  • dvc repro                           # Reproducir con DVC
  • docker compose up -d                # Levantar Airflow
  • pytest src/tests/                   # Ejecutar tests

🌐 Airflow UI: http

## 8. Conclusiones y Próximos Pasos

### ✅ Logros del Proyecto

**1. Aprendizaje No Supervisado:**
- ✓ Implementación de múltiples algoritmos de clustering (KMeans, DBSCAN, Agglomerative, GMM)
- ✓ Reducción dimensional con PCA, t-SNE y UMAP
- ✓ Detección de anomalías con Isolation Forest
- ✓ Segmentación de clientes exitosa

**2. Aprendizaje Supervisado:**
- ✓ Modelos de regresión para predecir delays de entrega
- ✓ Modelos de clasificación para categorizar órdenes
- ✓ Feature engineering enriquecido con cluster labels
- ✓ Métricas competitivas en todos los modelos

**3. MLOps & Infraestructura:**
- ✓ Pipeline completo orquestado con Kedro + Airflow
- ✓ Versionado de datos y modelos con DVC
- ✓ Containerización con Docker
- ✓ Reproducibilidad garantizada

---

### 💼 Valor de Negocio

- **Segmentación de Clientes**: Identificación de segmentos con características distintivas
- **Optimización Logística**: Insights sobre delays por cluster
- **Detección de Anomalías**: Sistema para identificar casos atípicos (fraude/VIP)
- **Modelos Predictivos**: Capacidad de predecir comportamiento futuro

---

### 🚀 Próximos Pasos

**1. Deployment:**
- Exponer modelos vía API REST (FastAPI)
- Dashboard de monitoreo en tiempo real (Streamlit/Dash)
- Integración con sistemas de CRM/ERP

**2. Mejora Continua:**
- Re-entrenamiento periódico con datos frescos
- A/B testing de estrategias por segmento
- Incorporar feedback del equipo de negocio

**3. Escalabilidad:**
- Migrar a cloud (AWS/GCP/Azure)
- Implementar feature store
- MLflow para experiment tracking
- CI/CD con GitHub Actions

---

### 📂 Artefactos Generados

✅ **Notebooks**: 6 notebooks completos (01-06)  
✅ **Modelos**: Guardados en `data/06_models/`  
✅ **Métricas**: Disponibles en `data/07_model_output/`  
✅ **Reportes**: Visualizaciones en `data/08_reports/`  
✅ **DAGs**: Pipelines orquestados en `airflow/dags/`  
✅ **Documentación**: `README.md` + `docs/`  

---

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import cross_val_score

import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


## 1. Carga de Datos y Resultados Previos

In [4]:
# Rutas de datos
root = Path('..')
data_dir = root / 'data'
model_input_dir = data_dir / '05_model_input'
model_output_dir = data_dir / '07_model_output'
reports_dir = data_dir / '08_reports'

# Cargar datos principales
model_input = pd.read_csv(model_input_dir / 'model_input.csv')
cluster_labels = pd.read_csv(model_input_dir / 'cluster_labels.csv', index_col=0)

# Cargar métricas de modelos supervisados
regression_metrics = pd.read_csv(model_output_dir / 'regression_metrics.csv')
classification_metrics = pd.read_csv(model_output_dir / 'classification_metrics.csv')

# Cargar métricas de clustering
clustering_metrics = pd.read_csv(model_output_dir / 'clustering_metrics.csv')

# Cargar anomalías
anomaly_scores = pd.read_csv(model_output_dir / 'anomaly_scores.csv', index_col=0)

print(f"📊 Datos cargados:")
print(f"  - Model Input: {model_input.shape}")
print(f"  - Cluster Labels: {cluster_labels.shape}")
print(f"  - Regression Models: {len(regression_metrics)}")
print(f"  - Classification Models: {len(classification_metrics)}")
print(f"  - Clustering Algorithms: {len(clustering_metrics)}")
print(f"  - Anomalies Detected (consensus): {anomaly_scores['is_anomaly_consensus'].sum()}")

📊 Datos cargados:
  - Model Input: (89316, 11)
  - Cluster Labels: (10000, 3)
  - Regression Models: 6
  - Classification Models: 6
  - Clustering Algorithms: 4
  - Anomalies Detected (consensus): 900


## 2. Integración: Clusters como Features

### 2.1 Enriquecimiento del Dataset

In [7]:
# Combinar model_input con cluster labels
data_enriched = model_input.copy()

# Agregar etiquetas de clusters (kmeans_label está en el índice de cluster_labels)
# Evitar errores si el largo no coincide con data_enriched
if len(cluster_labels) == len(data_enriched):
    data_enriched['cluster_kmeans'] = cluster_labels.index.values
else:
    # Si no hay alineación, inicializamos con NaN y avisamos
    data_enriched['cluster_kmeans'] = np.nan
    print(f"⚠️ cluster_labels ({len(cluster_labels)}) no coincide con data_enriched ({len(data_enriched)}). Se inicializa 'cluster_kmeans' con NaN.")

# Agregar scores de anomalías:
# - 'anomaly_score_iso' está en el índice de anomaly_scores
# - usar 'is_anomaly_consensus' como flag de anomalía
if len(anomaly_scores) == len(data_enriched):
    data_enriched['anomaly_score'] = anomaly_scores.index.values
    data_enriched['is_anomaly'] = anomaly_scores['is_anomaly_consensus'].values
else:
    # Si no hay alineación, inicializamos seguros
    data_enriched['anomaly_score'] = np.nan
    data_enriched['is_anomaly'] = False
    print(f"⚠️ anomaly_scores ({len(anomaly_scores)}) no coincide con data_enriched ({len(data_enriched)}). Se inicializan 'anomaly_score' con NaN y 'is_anomaly' con False.")

print(f"✅ Dataset enriquecido: {data_enriched.shape}")
print(f"\nNuevas columnas agregadas:")
print([col for col in data_enriched.columns if col.startswith('cluster_') or col.startswith('anomaly')])

⚠️ cluster_labels (10000) no coincide con data_enriched (89316). Se inicializa 'cluster_kmeans' con NaN.
⚠️ anomaly_scores (10000) no coincide con data_enriched (89316). Se inicializan 'anomaly_score' con NaN y 'is_anomaly' con False.
✅ Dataset enriquecido: (89316, 14)

Nuevas columnas agregadas:
['cluster_kmeans', 'anomaly_score']


### 2.2 Re-entrenamiento de Modelos con Clusters

In [9]:
# Preparar features con clusters
feature_cols = [col for col in data_enriched.columns if col not in ['delivery_delay', 'is_anomaly']]
target_regression = 'delivery_delay'

# Verificar que el target existe
if target_regression not in data_enriched.columns:
    print(f"⚠️ Target '{target_regression}' no encontrado. Usando columnas disponibles.")
    numeric_cols = data_enriched.select_dtypes(include=[np.number]).columns
    print(f"Columnas numéricas disponibles: {numeric_cols.tolist()}")
else:
    # Filtrar solo features numéricas válidas
    X_enriched = data_enriched[feature_cols].select_dtypes(include=[np.number])
    y_regression = data_enriched[target_regression]
    
    # Eliminar filas con NaN
    valid_idx = ~(X_enriched.isna().any(axis=1) | y_regression.isna())
    X_enriched = X_enriched[valid_idx]
    y_regression = y_regression[valid_idx]
    
    print(f"\n📈 Training Random Forest Regressor con clusters...")
    print(f"  - Features: {X_enriched.shape[1]}")
    print(f"  - Samples: {len(X_enriched)}")
    
    if len(X_enriched) < 2:
        print("⚠️ No hay suficientes muestras válidas para cross-validation (se requieren al menos 2).")
        print("   Revise que 'cluster_kmeans' y 'anomaly_score' no sean NaN y que 'delivery_delay' exista.")
    else:
        # Ajustar dinámicamente el número de folds
        cv_folds = min(5, len(X_enriched))
        
        # Modelo baseline (sin clusters)
        baseline_features = [col for col in X_enriched.columns if not col.startswith('cluster_')]
        X_baseline = X_enriched[baseline_features]
        
        rf_baseline = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        scores_baseline = cross_val_score(rf_baseline, X_baseline, y_regression, cv=cv_folds, scoring='r2')
        
        # Modelo enriquecido (con clusters)
        rf_enriched = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        scores_enriched = cross_val_score(rf_enriched, X_enriched, y_regression, cv=cv_folds, scoring='r2')
        
        print(f"\n🎯 Resultados de Cross-Validation (R² Score):")
        print(f"  - Baseline (sin clusters):     {scores_baseline.mean():.4f} ± {scores_baseline.std():.4f}")
        print(f"  - Enriquecido (con clusters):  {scores_enriched.mean():.4f} ± {scores_enriched.std():.4f}")
        
        improvement = ((scores_enriched.mean() - scores_baseline.mean()) / abs(scores_baseline.mean())) * 100 if scores_baseline.mean() != 0 else np.nan
        print(f"\n✅ Mejora relativa: {improvement:.2f}%")


📈 Training Random Forest Regressor con clusters...
  - Features: 6
  - Samples: 0
⚠️ No hay suficientes muestras válidas para cross-validation (se requieren al menos 2).
   Revise que 'cluster_kmeans' y 'anomaly_score' no sean NaN y que 'delivery_delay' exista.


## 3. Comparación Global de Modelos

### 3.1 Métricas de Regresión

In [ ]:
# Visualizar métricas de regresión
print("\n📊 Regression Models Performance:")
print("="*80)
display(regression_metrics)

# Gráfico de barras comparativo
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('R² Score', 'RMSE', 'MAE')
)

metrics_to_plot = ['r2', 'rmse', 'mae']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, metric in enumerate(metrics_to_plot, 1):
    if metric in regression_metrics.columns:
        fig.add_trace(
            go.Bar(
                x=regression_metrics['model'],
                y=regression_metrics[metric],
                marker_color=colors[i-1],
                showlegend=False
            ),
            row=1, col=i
        )

fig.update_layout(height=400, template='plotly_white', title_text='Regression Models Comparison')
fig.show()

# Mejor modelo
best_model_idx = regression_metrics['r2'].idxmax()
best_model = regression_metrics.loc[best_model_idx]
print(f"\n🏆 Mejor modelo de regresión: {best_model['model']} (R²={best_model['r2']:.4f})")

### 3.2 Métricas de Clasificación

In [ ]:
# Visualizar métricas de clasificación
print("\n📊 Classification Models Performance:")
print("="*80)
display(classification_metrics)

# Gráfico de barras comparativo
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Accuracy', 'Precision', 'Recall')
)

class_metrics = ['accuracy', 'precision', 'recall']
colors = ['#d62728', '#9467bd', '#8c564b']

for i, metric in enumerate(class_metrics, 1):
    if metric in classification_metrics.columns:
        fig.add_trace(
            go.Bar(
                x=classification_metrics['model'],
                y=classification_metrics[metric],
                marker_color=colors[i-1],
                showlegend=False
            ),
            row=1, col=i
        )

fig.update_layout(height=400, template='plotly_white', title_text='Classification Models Comparison')
fig.show()

# Mejor modelo
if 'accuracy' in classification_metrics.columns:
    best_clf_idx = classification_metrics['accuracy'].idxmax()
    best_clf = classification_metrics.loc[best_clf_idx]
    print(f"\n🏆 Mejor modelo de clasificación: {best_clf['model']} (Accuracy={best_clf['accuracy']:.4f})")

### 3.3 Métricas de Clustering

In [ ]:
# Visualizar métricas de clustering
print("\n📊 Clustering Algorithms Performance:")
print("="*80)
display(clustering_metrics)

# Gráfico de barras comparativo
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Silhouette Score', 'Davies-Bouldin Index', 'Calinski-Harabasz Score')
)

cluster_metrics = ['silhouette', 'davies_bouldin', 'calinski_harabasz']
colors = ['#e377c2', '#7f7f7f', '#bcbd22']

for i, metric in enumerate(cluster_metrics, 1):
    if metric in clustering_metrics.columns:
        fig.add_trace(
            go.Bar(
                x=clustering_metrics['algorithm'],
                y=clustering_metrics[metric],
                marker_color=colors[i-1],
                showlegend=False
            ),
            row=1, col=i
        )

fig.update_layout(height=400, template='plotly_white', title_text='Clustering Algorithms Comparison')
fig.show()

# Mejor algoritmo
best_cluster_idx = clustering_metrics['silhouette'].idxmax()
best_cluster = clustering_metrics.loc[best_cluster_idx]
print(f"\n🏆 Mejor algoritmo de clustering: {best_cluster['algorithm']} (Silhouette={best_cluster['silhouette']:.4f})")